# Reef Discovery Phase 3: U-Net Training
This notebook trains a PyTorch U-Net model on the 128x128px patches extracted from the deterministic pipeline.

In [ ]:
!pip install torch torchvision segmentation-models-pytorch albumentations tqdm

In [ ]:
import os
import sys
import shutil
from glob import glob
import torch
from torch.utils.data import DataLoader, random_split
import torch.optim as optim
from tqdm import tqdm

# Add project root to path
sys.path.append(os.path.abspath('..'))

from src.ml_unet_dataset import ReefPatchDataset, get_training_augmentation, get_validation_augmentation
from src.ml_unet_model import ReefUNet, get_loss_function, calculate_iou


### 1. Aggregate Dataset
We aggregate patches from all `reef_output_*/ml_dataset` folders into a master folder.

In [ ]:
MASTER_DATASET = '../data/master_ml_dataset'
os.makedirs(f'{MASTER_DATASET}/images', exist_ok=True)
os.makedirs(f'{MASTER_DATASET}/masks', exist_ok=True)

patch_dirs = glob('../reef_output_*/ml_dataset')
idx = 0
for pdir in patch_dirs:
    images = glob(os.path.join(pdir, 'images', '*.tif'))
    for img_path in images:
        mask_path = img_path.replace('images', 'masks')
        if os.path.exists(mask_path):
            shutil.copy(img_path, f'{MASTER_DATASET}/images/patch_{idx:04d}.tif')
            shutil.copy(mask_path, f'{MASTER_DATASET}/masks/patch_{idx:04d}.tif')
            idx += 1
print(f'Total aggregated patches: {idx}')

### 2. Setup DataLoaders

In [ ]:
# Create full dataset
full_dataset = ReefPatchDataset(
    images_dir=f'{MASTER_DATASET}/images',
    masks_dir=f'{MASTER_DATASET}/masks',
    transform=get_training_augmentation()
)

# Split: 70% Train, 30% Val
train_size = int(0.7 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset, [train_size, val_size], 
    generator=torch.Generator().manual_seed(42)
)

# Override val_dataset transform to None
val_dataset.dataset.transform = get_validation_augmentation()

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")

### 3. Initialize Model and Training Loop

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = ReefUNet().to(device)
criterion = get_loss_function()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
epochs = 25

best_iou = 0.0
os.makedirs('../models', exist_ok=True)

for epoch in range(epochs):
    model.train()
    train_loss = 0
    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        images, masks = images.to(device), masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    # Validation
    model.eval()
    val_loss, val_iou = 0, 0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, masks).item()
            val_iou += calculate_iou(outputs, masks)
            
    val_iou /= len(val_loader)
    print(f"Epoch {epoch+1} - Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f} | Val IoU: {val_iou:.4f}")
    
    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), '../models/unet_reef_best.pth')
        print("  -> Saved new best model!")
        
print("Training Complete!")